# 6. Agent的高级用法1：设置Agent名称

创建Agent时，LangChain允许用户指定其名称

## 6.1 用法

In [1]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from rich import print as rprint
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.tools import BaseTool
from langchain.messages import SystemMessage, HumanMessage

load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model= ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

agent = create_agent(
    model=model,
    name="chat_assistant"
)
response = agent.invoke({"messages": ["你好"]})
# rprint(response)
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================
Name: chat_assistant

你好！有什么我可以帮你处理的？


In [2]:
rprint(response)

{
    'messages': [
        HumanMessage(
            content='你好',
            additional_kwargs={},
            response_metadata={},
            id='903e0faa-e67f-4ce0-930f-6be48b0df04b'
        ),
        AIMessage(
            content='你好！有什么我可以帮你处理的？',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 14,
                    'prompt_tokens': 4389,
                    'total_tokens': 4403,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 3840,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna',
                'system_fingerprint': None,
                'id': 'resp_0fcb76e31a692b64016aba1267989487d2848b716175540830',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            name='chat_assistant',
            id='lc_run--01a0e6d7-ccc6-7e31-8c5f-e55e2e403eb2-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 4389,
                'output_tokens': 14,
                'total_tokens': 4403,
                'input_token_details': {'cache_read': 3840},
                'output_token_details': {}
            }
        )
    ]
}

**name 具体做了什么？** 源码（`langchain/agents/factory.py`）里一共有三处用到它：

1. 作为这张图的名字：`agent.name`。以后把这个 Agent 作为子图放进更大的图时，用的就是这个名字
2. `model` 节点每次拿到模型的回复后，执行 `output.name = name`，所以返回的 `AIMessage` 带上了 `name='chat_assistant'`。上面 `pretty_print` 输出的 `Name: chat_assistant` 就是它
3. 写进运行配置的 `metadata`（键名是 `lc_agent_name`），LangSmith 的 trace 里可以据此区分不同的 Agent

第 1、3 处可以直接查看（不请求模型）：

In [6]:
print("agent.name:", agent.name)
print("metadata:  ", agent.config["metadata"])

agent.name: chat_assistant
metadata:   {'ls_integration': 'langchain_create_agent', 'lc_agent_name': 'chat_assistant'}


## 6.2 经典使用场景

name 在 Multi-Agent 场景中最常被提及，用于区分不同的 Agent。但它的作用并不局限于多 Agent编排。在实际工程中，出现如下场景，通常都建议为 Agent 设置一个清晰且稳定的 name 。

1. 流式输出归因

在启用流式输出时，name 可用于 标识当前输出内容来自哪个 Agent 。

这在多 Agent 协作、Agent 嵌套调用，或前端需要实时展示不同执行主体输出时尤其有用，便于准确区分 token 或事件的来源。

2. 消息身份标记

设置 name 后，Agent 产生的 AIMessage 会携带对应的name信息 。

这使得系统在 保存会话记录 、 回放执行过程 、 构建审计日志 或 前端展示消息角色 时，能够明确识别消息的生成者。

3. 调试与trace可读性

在调试、日志分析和链路追踪过程中，name 可以作为 Agent 的稳定标识，帮助开发者 快速判断当前执行的是哪个 Agent 。

当系统中存在多个能力相近的 Agent，或一个 Agent 被嵌套在更复杂的工作流中时，名称能够显著提升trace 的可读性 和 问题定位 效率。

4. 组件化封装

在工程实践中，Agent 常被封装为可复用的能力模块，例如检索助手、SQL 助手、报告生成助手等。

为 Agent 设置 name ，有助于在模块注册、运行监控、日志归档和能力复用时保持一致的身份标识。

如果后续需要将该 Agent 进一步作为子图节点、工具能力或子模块接入更复杂系统，也能降低维护和迁
移成本。

# 7. Agent的高级用法2：系统提示词
使用 create_agent 创建 Agent 时，需传入 模型 和 工具 、可选地传入 系统提示词 。提示词为Agent提供了任务背景、行为准则和操作指南。

系统指令，即SystemMessage，通过 system_prompt 设置，定义 Agent 行为。这个参数可以是 str或者 SystemMessage类型 。

使用建议：
* 明确说明 Agent 的角色
* 定义输出格式
* 说明何时使用工具

提示词设置有两种方式： 静态设置 和 动态设置 。动态设置需要借助中间件，后续讲解。

In [3]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model= ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

# 2.导入工具
web_search = TavilySearch(max_results=2)
# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)
# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)
print(result['messages'][-1].content)

[{'type': 'text', 'text': '2026年足球世界杯将由**加拿大、墨西哥和美国三个国家联合举办**。\n\n这是世界杯首次由三个国家共同承办，比赛将在三国的多个城市举行。  \n来源： [国际足联 FIFA](https://www.fifa.com/en/articles/fifa-world-cup-2026-hosts-cities-dates-usa-mexico-canada)', 'annotations': [], 'id': 'msg_04847f6a31898b8f016aba1505ea9087d0a693ff0a1623309b', 'phase': 'final_answer'}]


In [4]:
rprint("===============================================")
rprint(result)

===============================================

{
    'messages': [
        HumanMessage(
            content='请帮我查询2026年足球世界杯是哪个国家举办的？',
            additional_kwargs={},
            response_metadata={},
            id='f77e3b70-0446-4fc7-ba61-6d78927ab1f2'
        ),
        AIMessage(
            content=[
                {
                    'arguments': '{"query":"2026 FIFA World Cup host countries official 
FIFA","include_domains":["fifa.com"],"exclude_domains":[],"search_depth":"basic","include_images":false,"time_range
":null,"topic":"general","start_date":null,"end_date":null}',
                    'call_id': 'call_3LN5Qp1vsff8j4rFhokY9tCl',
                    'name': 'tavily_search',
                    'type': 'function_call',
                    'id': 'fc_04847f6a31898b8f016aba14fe9d4c87d0bc356a3423dd9357',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'create_time': 1790579965.785751,
                        'turn_id': '01a0e6e1-fd1e-7181-869e-8a0d468c6329'
                    },
                    'metadata': {'turn_id': '01a0e6e1-fd1e-7181-869e-8a0d468c6329'}
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_04847f6a31898b8f016aba14fd91fc87d09a979868177d864a',
                'created_at': 1790579965.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_04847f6a31898b8f016aba14fd91fc87d09a979868177d864a',
            tool_calls=[
                {
                    'name': 'tavily_search',
                    'args': {
                        'query': '2026 FIFA World Cup host countries official FIFA',
                        'include_domains': ['fifa.com'],
                        'exclude_domains': [],
                        'search_depth': 'basic',
                        'include_images': False,
                        'time_range': None,
                        'topic': 'general',
                        'start_date': None,
                        'end_date': None
                    },
                    'id': 'call_3LN5Qp1vsff8j4rFhokY9tCl',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1284,
                'output_tokens': 69,
                'total_tokens': 1353,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='{"query": "2026 FIFA World Cup host countries official FIFA", "follow_up_questions": null, 
"answer": null, "images": [], "results": [{"url": "https://store.fifa.com", "title": "Official FIFA Store - The 
Home of Official FIFA World Cup 2026...", "content": "Sale\\n\\n### FIFA World Cup 2026™ Host Country Canada 
Scarf\\n\\nSale price  €17,50 EUR Regular price   ~~€35,00 EUR~~ -50%\\n\\nRegular price €17,50 EUR\\n\\n2026 World
Cup 8\\n\\nFIFA World Cup 2026™ 8\\" USA Squishmallows Plush Mascot\\n\\nSaleExclusive\\n\\n### FIFA World Cup 
2026™ 8\\" USA Squishmallows Plush Mascot\\n\\nSale price  €15,00 EUR Regular price   ~~€30,00 EUR~~ 
-50%\\n\\nRegular price €15,00 EUR\\n\\nFIFA World Cup 2026™ Brazil Sublimated Scarf\\n\\nFIFA World Cup 2026™ 
Brazil Sublimated Scarf\\n\\nSaleNew [...] UP TO 50% OFF\\n FIFA Women\'s World Cup 2027™\\n Nations  \\n  + 
Algeria\\n  + Argentina\\n  + Austria\\n  + Belgium\\n  + Brazil\\n  + Canada\\n  + Colombia\\n  + Cote 
d\'Ivoire\\n  + Curaçao\\n  + Czechia\\n  + Ecuador\\n  + Egypt\\n  + England\\n  + France\\n  + Germany\\n  + 
Ghana\\n  + Iraq\\n  + Japan\\n  + Mexico\\n  + Netherlands\\n  + New Zealan

上面 `print(result['messages'][-1].content)` 打印出来的是一个列表：带了工具时，这个中转模型走的是 Responses API，`content` 是内容块列表。只想要文字，用 `result['messages'][-1].text`（见 02 笔记 4.3.3）。

系统提示词不会在消息列表中。为什么？见 7.1。

In [5]:
# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"

# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers], # 工具列表
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
)

response = agent.invoke(
    {"messages": [
    {"role": "user", "content": "10加上20再加上30是多少？"}
    ]},
    )
rprint(response)

{
    'messages': [
        HumanMessage(
            content='10加上20再加上30是多少？',
            additional_kwargs={},
            response_metadata={},
            id='f9490f2d-b684-48cd-90d2-4660697f62c1'
        ),
        AIMessage(
            content=[
                {
                    'id': 'rs_0521093072d0a8e1016aba177c442087d0a00011afd5e52751',
                    'summary': [],
                    'type': 'reasoning',
                    'content': [],
                    'encrypted_content': 
'gAAAAABquhd-9gS66Pu7UYMwX60uEjdfYTLmpPw1ldemB5173zbT9WNbKVMQsLIoZArvnHHG5bZthq4_pI4dxWgdEkmSfeBQf-XB8Kkj0ZU7MYKC08
HC8vn61vsoBNaNGai4X9s8bbOVujKrBHSQimz-mp0nX28vjSqLwE-Kr_D0UGL59gOsjzuP2xGpYL7U-_Y8SRS91HivkvOO-nynkpLbJkVyCveaimvA2
pvfw51j5KiIxkcMaPCzhAWmAtacSwO2ONyRvbjt5kGj99jn0q66AjUM7DDc0cB1xEBtWpzLtHRsM0TwGrB84B8vpQXiqXXY0mo205rB4Xyp25mDVqSD
ABl-5lQXBuV8X6JeCSOeRwrTcdTwxTyPJiaWlCeLvolwqaajhFGdOMSW0G3GsKFqGb0k_IV6rmpJNZHiLvGSWR1RmGhUXLO08nQYKQhZxxRoG8xq5t-
HZGngc1W5EjNSNLV6wb79pjQMKARv8mOjD8F3HF-YuEApJyMcIT-CRdJ0e2CR7R8qiFnOTzW9C9ucdeZdrDmdKVPnHuGTe_J9yds0lxHeDdSJ1wk6mY
tQiTTPkz2wfEMEqpsrOZ92CB0YAgsS0YPA78SRSl8iiZhAMfVM1Z2KedxPTTwngGZ0zQ_z8Vo_K8VsSETIsKg3SUdQ12dVaKBFm2dUdvCto1Wvqr5cf
EksLW__Qcz_KftXyYjeRgmD80_XuPG3CbMAQZQsBY98XzYUg8Mfm4VTEZnJ9mDwHZfBJ7HuIii1JT6VaIKUjx8QU1Tlumw_vlnPNPMCp5fmuVEMFTdV
XivRWK3X9T0wJsaoQG41-5a9gHi0OLPyFFtuM8qcYPX1DSZ8WqGJc3o1doioTJBm9RmxeujZ3hg-rVhAR166O1PvSavmDVA_qicKwfXV0jJ6GNDo6_4
KMxx-DqE_WLeRn82TSjF57xnw1Xs8osNXhrB66syUtB5zOFNkRoJ6BlSB2Wbq5PSN8JP5GsZBeSIkcfX7z8SD2rXriBN1nXDqWUz_JwTEJ-maY4D8bb
CR1bFkDuJ1H5lb4tMcoOxPAaqfPqkK4JbfCkTzn63j9qu57TONyTl8bGDNLcJtZC2ogUSJYjIQXA4ng6wUy0uIXg9-tzTZ_Bmwk0UFavdYrCaIvh6vh
E-6WV66xSEaqNb67-dqUc8PefYViNBz0TMDayJJgQe3vucfgD3BPDWsCqpBTglJ-prFxSiGrRIJEhiZjMjkkCZuTWueAqjm3vhgAnhL0EUzNV3trLeV
fyz_Q3Nip2wjsjxkYrrHd7OqM1SrT_VQMJdjR_iF6xh1UNJj_mmEolOgbJ-nASltVMH7mWOb7g-bgyhb6YV927YxYLRIAXlDZ-c8wnygq-MjfrPGyUa
wF5mHfNv1A0moMT1dhgbuIirBV4N6SBp8Gr_QumXAhE3yIiRCGntkpZ0wneg71ph__G0VYamOnCclF62lVZULMU-5O6vhJDSfwB6HEbcSHgOmG0m5Ud
2VX0YlhTUcYQ==',
                    'status': 'completed',
                    'internal_chat_message_metadata_passthrough': {
                        'turn_id': '01a0e6eb-9409-7850-964a-02410047d2fd'
                    },
                    'metadata': {'turn_id': '01a0e6eb-9409-7850-964a-02410047d2fd'}
                },
                {
                    'type': 'text',
                    'text': '60。',
                    'annotations': [],
                    'id': 'msg_0521093072d0a8e1016aba177e545c87d09db6212f4c2df7f0',
                    'phase': 'final_answer'
                }
            ],
            additional_kwargs={},
            response_metadata={
                'id': 'resp_0521093072d0a8e1016aba177206d087d0b8307f0e1f6371c6',
                'created_at': 1790580594.0,
                'metadata': {},
                'model': 'gpt-6-luna',
                'object': 'response',
                'service_tier': 'default',
                'status': 'completed',
                'model_provider': 'openai',
                'model_name': 'gpt-6-luna'
            },
            id='resp_0521093072d0a8e1016aba177206d087d0b8307f0e1f6371c6',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 154,
                'output_tokens': 46,
                'total_tokens': 200,
                'input_token_details': {'cache_creation': 0, 'cache_read': 0},
                'output_token_details': {'reasoning': 38}
            }
        )
    ]
}

注意这次模型**没有调用** `add_numbers`，直接回答了"60。"（`tool_calls=[]`）。系统提示词只说了"你是一个数学助手"，没有说要用工具；而且 `add_numbers` 一次只能加两个数，模型自己心算反而更快。

这正好对应第 7 节开头的使用建议"说明何时使用工具"：想让 Agent 用工具，就要在系统提示词里写清楚。7.1.2 会改一下系统提示词，再运行一次。

## 7.1 为什么返回的消息列表里看不到系统提示词？

### 7.1.1 返回的是状态，而系统提示词不在状态里

`agent.invoke` 返回的是**运行结束时的状态**（01 笔记 2.4）。状态里的 `messages` 只有两个来源：

1. 调用时传进来的消息，也就是 `invoke` 的输入
2. 各个节点运行后**返回**的消息：`model` 节点返回 `AIMessage`，`tools` 节点返回 `ToolMessage`，它们由 `add_messages` 追加到 `messages` 的末尾

`system_prompt` 不属于其中任何一种。它的处理分两步（源码 `langchain/agents/factory.py`，有删减）：

```python
# ① 创建 Agent 时：字符串转成 SystemMessage，保存在 create_agent 内部的变量里，而不是状态里
if isinstance(system_prompt, SystemMessage):
    system_message = system_prompt
else:
    system_message = SystemMessage(content=system_prompt)

# ② model 节点每次运行时：
request = ModelRequest(system_message=system_message, messages=state["messages"], ...)
messages = request.messages                          # 状态里的消息
if request.system_message:
    messages = [request.system_message, *messages]   # 临时拼到最前面，只用于这一次调用
output = model_.invoke(messages)                     # 模型收到的是带系统消息的列表
...
return {"messages": [output]}                        # 写回状态的只有模型的回复（简化后的写法）
```

所以：**模型每次都能收到系统提示词，但它从来没有被写进状态**，返回值里自然看不到。

### 7.1.2 验证：看看模型每次实际收到了什么

返回值里看不到，可以在"调用模型之前"看。LangChain 的**回调**（callback）可以做到：`on_chat_model_start` 会在每次调用模型之前触发，参数 `messages` 就是模型实际收到的消息。回调通过 `config={"callbacks": [...]}` 传入，Agent 内部调用模型时也会触发它。

顺便按第 7 节开头的建议，在系统提示词里写清楚"何时使用工具"，还是问 `10加上20再加上30是多少？`（会请求模型几次，取决于模型调用几次工具）：

In [7]:
from langchain_core.callbacks import BaseCallbackHandler

def brief(msg):
    """把一条消息缩成一行，方便打印"""
    if getattr(msg, "tool_calls", None):
        return "调用工具 " + "、".join(f"{tc['name']}{tc['args']}" for tc in msg.tool_calls)
    return msg.text[:40]

class ShowModelInput(BaseCallbackHandler):
    """每次调用模型之前，打印模型实际收到的消息"""
    def __init__(self):
        self.count = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.count += 1
        print(f"—— 第 {self.count} 次调用模型，收到 {len(messages[0])} 条消息 ——")
        for msg in messages[0]:
            print(f"    {type(msg).__name__:<14} {brief(msg)}")

math_agent = create_agent(
    model=model,
    tools=[add_numbers],
    system_prompt="你是一个数学助手。做加法时必须调用 add_numbers 工具，不要自己心算；工具每次只能把两个数相加。",
)
response = math_agent.invoke(
    {"messages": [{"role": "user", "content": "10加上20再加上30是多少？"}]},
    config={"callbacks": [ShowModelInput()]},
)

print("\n—— 返回的状态里的消息 ——")
for msg in response["messages"]:
    print(f"    {type(msg).__name__:<14} {brief(msg)}")

—— 第 1 次调用模型，收到 2 条消息 ——
    SystemMessage  你是一个数学助手。做加法时必须调用 add_numbers 工具，不要自己心算；
    HumanMessage   10加上20再加上30是多少？


—— 第 2 次调用模型，收到 4 条消息 ——
    SystemMessage  你是一个数学助手。做加法时必须调用 add_numbers 工具，不要自己心算；
    HumanMessage   10加上20再加上30是多少？
    AIMessage      调用工具 add_numbers{'a': 10, 'b': 20}
    ToolMessage    和为：30


—— 第 3 次调用模型，收到 6 条消息 ——
    SystemMessage  你是一个数学助手。做加法时必须调用 add_numbers 工具，不要自己心算；
    HumanMessage   10加上20再加上30是多少？
    AIMessage      调用工具 add_numbers{'a': 10, 'b': 20}
    ToolMessage    和为：30
    AIMessage      调用工具 add_numbers{'a': 30, 'b': 30}
    ToolMessage    和为：60



—— 返回的状态里的消息 ——
    HumanMessage   10加上20再加上30是多少？
    AIMessage      调用工具 add_numbers{'a': 10, 'b': 20}
    ToolMessage    和为：30
    AIMessage      调用工具 add_numbers{'a': 30, 'b': 30}
    ToolMessage    和为：60
    AIMessage      10 + 20 + 30 = 60。


对比两部分输出：

- 模型一共被调用了 3 次，每次收到的消息**第一条都是 `SystemMessage`**，后面跟着状态里已有的消息，而且一次比一次长
- 返回的状态里**没有 `SystemMessage`**，只有用户消息、模型的回复和工具结果
- 这次模型按系统提示词的要求调用了两次 `add_numbers`：先算 10 + 20，再算 30 + 30，最后给出答案

开启了 LangSmith 时，也可以在 trace 里点开 `model` 这一步，看到模型收到的完整消息。

### 7.1.3 为什么要这样设计

系统提示词是 Agent 的**配置**，而状态记录的是**对话本身**。把两者分开有几个好处：

1. **历史干净**：状态里只有用户和 Agent 之间真正发生过的消息，保存、展示、回放时都不用再过滤系统消息
2. **配合记忆不会重复**：用 `checkpointer` 保存多轮对话时（01 笔记 2.5.5），系统提示词不会随每一轮都存一遍
3. **修改后马上生效**：系统提示词不随历史保存，改了提示词、重新创建 Agent 后，使用同一个 `checkpointer` 的旧会话，下一次调用就会用上新的提示词
4. **方便动态修改**：`model` 节点每次都会新建一个请求对象，系统提示词只是其中的 `system_message` 字段。第 7 节开头提到的"动态设置"，就是用中间件在调用模型之前改写这个字段（比如根据用户身份换一段提示词），同样不会写进历史

### 7.1.4 system_prompt 与 messages 里的 system 消息

02 笔记第 3 节用过另一种写法：把 `{"role": "system", ...}` 直接放进 `messages`。两者的区别：

| | `create_agent(system_prompt=...)` | `messages` 里的 system 消息 |
|---|---|---|
| 在哪里设置 | 创建 Agent 时设置一次 | 每次 `invoke` 时随输入传入 |
| 模型能否收到 | 能，每次调用模型时拼在最前面 | 能，它就在状态的消息列表里 |
| 是否出现在返回值里 | 否 | 是 |
| 配合 `checkpointer` | 不会重复保存 | 每轮都传，就会在历史里堆积多条 |
| 适合 | 固定的角色、规则、输出格式 | 只对某一次调用生效的临时指令 |

两者同时使用时，模型会收到两条系统消息：`system_prompt` 在最前面，`messages` 里的那条排在它后面。

## 7.2 小结

- `name`：给 Agent 起名。它会成为图的名字，写进每条 `AIMessage` 的 `name`，也会出现在 trace 的 metadata 里，便于在多 Agent 协作、日志和流式输出中区分消息的来源
- `system_prompt`：可以是字符串或 `SystemMessage`。它保存在 Agent 内部，只在每次调用模型时临时拼到消息列表的最前面，**不会写进状态**，所以返回的 `messages` 里看不到；想确认模型收到了它，可以用回调，或者在 LangSmith 的 trace 里查看
- 写系统提示词时，除了角色，还要写清楚输出格式和**何时使用工具**：只写"你是数学助手"时，模型选择了自己心算；写明"必须调用 add_numbers"之后，它才会使用工具